In [ ]:
import torch
import torch.nn as nn

In [ ]:
d_model = 512
num_heads = 8
d_ff = 2048
vocab_size = 10000
max_seq_len = 100

# 1. Implementation of Token Embedding

In [ ]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embeddings = nn.Embedding(vocab_size, d_model)

  def forward(self, x):
    return self.embeddings(x)

In [ ]:
vocab_size = 10
d_model = 5
model = Embedding(vocab_size, d_model)
x = torch.tensor([
    [1, 5, 2, 8, 3],
    [4, 7, 6, 2, 1]
])
output = model(x)

print(output.shape)
print(output)

torch.Size([2, 5, 5])
tensor([[[-0.7285,  0.3655, -1.1973,  0.2297,  0.8160],
         [ 0.6650, -0.1784,  0.8152, -0.3253, -2.2140],
         [-1.2980,  0.2612,  0.2875, -0.8271,  0.5970],
         [ 0.2065,  1.1354,  1.1838, -1.0638,  0.2502],
         [-0.3625, -0.0469,  0.5957, -1.9940, -1.2089]],

        [[-1.1550,  0.7584,  0.3133, -0.4998, -0.5724],
         [-1.3458, -1.3537,  1.4910,  1.0993, -1.8988],
         [ 0.1559, -0.5161,  0.6357,  1.0547, -0.4373],
         [-1.2980,  0.2612,  0.2875, -0.8271,  0.5970],
         [-0.7285,  0.3655, -1.1973,  0.2297,  0.8160]]],
       grad_fn=<EmbeddingBackward0>)


# Implemenatation of Positional Encoding

In [ ]:
import math
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000):
    super().__init__()

    pe = torch.zeros(max_len, d_model)

    position = torch.arange(0, max_len).unsqueeze(1)

    div_term = torch.exp(torch.arange(0, d_model,2 ) * (- math.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0)

    self.register_buffer('pe', pe)
  def forward(self,x):
    return x +self.pe[:, :x.size(1)]

# Scaled dot-product Attention

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):

    d_k = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    weights = torch.softmax(scores, dim=-1)

    output = torch.matmul(weights, V)

    return output

# Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()

    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.Wq = nn.Linear(d_model, d_model)
    self.Wk = nn.Linear(d_model, d_model)
    self.Wv = nn.Linear(d_model, d_model)

    self.Wo = nn.Linear(d_model, d_model)

  def split_heads(self, x, batch_size):
    # batch, seq_len, d_model = x.size() # This line is not needed as batch_size is passed
    # The batch variable is already provided via batch_size, so we can directly use it
    _, seq_len, _ = x.size() # Use _ for unused variables
    x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
    return x.transpose(1,2)

  def combine_heads(self, x):

    batch, heads, seq_len, d_k = x.size()

    x = x.transpose(1,2)

    return x.contiguous().view(batch, seq_len, num_heads*d_k)



  def forward(self, x):
        batch_size = x.size(0) # Get batch size from input tensor x
        Q = self.split_heads(self.Wq(x), batch_size)
        K = self.split_heads(self.Wk(x), batch_size)
        V = self.split_heads(self.Wv(x), batch_size)

        attn = scaled_dot_product_attention(Q,K,V)

        output = self.combine_heads(attn)

        return self.Wo(output)

model = TransformerModel()

dummy_input = torch.randint(0, vocab_size, (2, 10))

output = model(dummy_input)

print(output.shape)

# Feed Forward Network

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.ReLU(),
        nn.Linear(d_ff, d_model)

    )

  def forward(self, x):
    return self.net(x)

# Encode Layer

In [ ]:
class EncodeLayer(nn.Module):
  def __init__(self,d_model, num_heads,d_ff):
    super().__init__()

    self.attn = MultiHeadAttention(d_model, num_heads)
    self.ff = FeedForward(d_model, d_ff)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

  def forward(self, x):

    x = self.norm1(x + self.attn(x))
    x = self.norm2(x + self.ff(x))

    return x

# Encoder Stack

In [ ]:
class Encoder(nn.Module):

    def __init__(self, num_layers):

        super().__init__()

        self.layers = nn.ModuleList(
            [EncodeLayer(d_model, num_heads, d_ff)
             for _ in range(num_layers)]
        )

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x

In [ ]:
class TransformerEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = Embedding(vocab_size, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder = Encoder(num_layers=6)

    def forward(self, x):

        x = self.embedding(x)

        x = self.pos_encoding(x)

        x = self.encoder(x)

        return x

In [ ]:
class TransformerModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = TransformerEncoder()

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.encoder(x)

        return self.fc(x)

torch.Size([2, 10, 10000])
